# Paper 3 backbone on YOLO26n: SAConv (Switchable Atrous Convolution) down-sampling

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Paper 3:** the stride-2 `Conv` at layers 3, 5, 7 replaced by **SAConv** |
| Neck FPN + PAN (layers 11–22) | YOLO26n base, unchanged |
| Head (Detect, layer 23) | YOLO26n base, unchanged (one-to-one, NMS-free) |

The baseline ("Standard YOLO26n Backbone") is **not trained here**; its values are already filled in the final table.

**Run this notebook in two steps (by hand):**
1. **Step A — `EPOCHS = 1` (≈ 10–15 min):** check that the model exports to NCNN and measure its FPS on the Raspberry Pi 5.
   Speed does not depend on the number of epochs, so this FPS is already the real FPS. Precision / mAP after 1 epoch are meaningless.
2. **Step B — `EPOCHS = 100`:** only if step A passes (see section 7), set `EPOCHS = 100` in section 4 and run the notebook again.

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**

## 1. Setup

In [ ]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

In [ ]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so saconv_backbone.py can be imported

## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [ ]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

## 3. Paper 3 backbone: SAConv

**What SAConv does.** The same 3×3 conv weight is applied with two atrous (dilation) rates:
rate 1 picks up fine texture, rate 3 sees a larger structure. A **switch** (5×5 average pool → 1×1 conv → sigmoid) decides,
for every position, how much each branch counts. A **pre-global context** (global average pool → 1×1 conv, added to the input) and a
**post-global context** (same, on the output) add a summary of the whole image. BatchNorm + SiLU follow.

$$y = S(x)\cdot \mathrm{Conv}(x, w, r{=}1) + \big(1 - S(x)\big)\cdot \mathrm{Conv}(x, w + \Delta w, r{=}3)$$

The rate-3 branch uses the shared weight `w` plus a small learnable difference `Δw` (starts at 0).

**Where it sits in YOLO26n.**

| Layer | Base YOLO26n | Paper 3 backbone | Output |
|---|---|---|---|
| 3 | Conv 3×3, stride 2 | **SAConv** | P3/8 (80 × 80) |
| 5 | Conv 3×3, stride 2 | **SAConv** | P4/16 (40 × 40) |
| 7 | Conv 3×3, stride 2 | **SAConv** | P5/32 (20 × 20) |

No layer index changes, so the neck connections stay the same.

**Difference to Paper 3.** Paper 3 replaced every down-sampling Conv in the backbone *and* the neck. Here only backbone layers 3, 5, 7
are replaced: the neck stays YOLO26n base, and layers 0 and 1 run at 320 × 320 and 160 × 160, where SAConv would slow a Raspberry Pi 5 down a lot.

**Pretrained start.** The shared weight `w` and the BatchNorm of each SAConv are copied from the pretrained `yolo26n.pt` Conv it replaces,
and both context convs start at zero, so the new layers start close to the pretrained model instead of from random weights.

**Export.** Before the NCNN export, the two shared-weight branches are turned into two plain conv layers (`to_static`),
so NCNN only sees standard layers. Params and GFLOPs are counted on this deploy form.

**Why it may fit the task.** People stand at different distances from the camera, so their size changes; the two atrous rates can cover both.
**Caution.** In Paper 3, SAConv increased the parameters by 31 % and its FPS was measured on a desktop GPU. The main question here is
whether it runs on the Raspberry Pi 5 and how fast.

In [ ]:
%%writefile saconv_backbone.py
"""Paper 3 backbone for YOLO26n: stride-2 Conv at layers 3, 5, 7 -> SAConv (Switchable Atrous Convolution)."""
import torch
import torch.nn as nn
import torch.nn.functional as F

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

SACONV_LAYERS = (3, 5, 7)


class SAConv(nn.Module):
    """Switchable Atrous Convolution: shared 3x3 weight at dilation 1 and 3, mixed by a learned switch, with pre/post global context."""

    def __init__(self, c1, c2, k=3, s=2, rates=(1, 3)):
        super().__init__()
        self.c1, self.c2, self.k, self.s, self.rates = c1, c2, k, s, rates
        self.pre_context = nn.Conv2d(c1, c1, 1, bias=True)
        self.post_context = nn.Conv2d(c2, c2, 1, bias=True)
        for m in (self.pre_context, self.post_context):  # start as identity
            nn.init.zeros_(m.weight)
            nn.init.zeros_(m.bias)
        self.switch = nn.Conv2d(c1, 1, 1, stride=s, bias=True)
        nn.init.zeros_(self.switch.weight)
        nn.init.zeros_(self.switch.bias)  # sigmoid(0) = 0.5: both branches equal at the start
        self.weight = nn.Parameter(torch.empty(c2, c1, k, k))
        nn.init.kaiming_uniform_(self.weight, a=5 ** 0.5)
        self.weight_diff = nn.Parameter(torch.zeros(c2, c1, k, k))  # extra weight of the large-rate branch
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU()
        self.static = False

    def to_static(self):
        """Replace the shared-weight branches by two plain Conv2d layers (same output), for NCNN export."""
        if self.static:
            return
        r1, r2 = self.rates
        p = self.k // 2
        self.conv_small = nn.Conv2d(self.c1, self.c2, self.k, self.s, p * r1, dilation=r1, bias=False)
        self.conv_large = nn.Conv2d(self.c1, self.c2, self.k, self.s, p * r2, dilation=r2, bias=False)
        with torch.no_grad():
            self.conv_small.weight.copy_(self.weight)
            self.conv_large.weight.copy_(self.weight + self.weight_diff)
        self.conv_small.to(self.weight.device, self.weight.dtype)
        self.conv_large.to(self.weight.device, self.weight.dtype)
        del self.weight, self.weight_diff
        self.static = True

    def forward(self, x):
        x = x + self.pre_context(F.adaptive_avg_pool2d(x, 1))
        s = torch.sigmoid(self.switch(F.avg_pool2d(x, 5, 1, 2)))
        if self.static:
            small, large = self.conv_small(x), self.conv_large(x)
        else:
            r1, r2 = self.rates
            p = self.k // 2
            small = F.conv2d(x, self.weight, None, self.s, p * r1, r1)
            large = F.conv2d(x, self.weight + self.weight_diff, None, self.s, p * r2, r2)
        y = s * small + (1 - s) * large
        y = y + self.post_context(F.adaptive_avg_pool2d(y, 1))
        return self.act(self.bn(y))


def add_saconv(model, layers=SACONV_LAYERS):
    """Replace the stride-2 Conv at the given layers with SAConv, starting from that Conv's (pretrained) weight and BN."""
    for i in layers:
        old = model.model[i]
        assert isinstance(old, Conv) and old.conv.kernel_size == (3, 3) and old.conv.stride == (2, 2), f"layer {i} is not a 3x3 s2 Conv"
        new = SAConv(old.conv.in_channels, old.conv.out_channels, 3, 2)
        with torch.no_grad():
            new.weight.copy_(old.conv.weight)
        new.bn.load_state_dict(old.bn.state_dict())
        new.i, new.f, new.type = old.i, old.f, "SAConv"  # keep the routing attributes used by the model's forward
        new.np = sum(p.numel() for p in new.parameters())
        model.model[i] = new
    return model


def to_static(model):
    """Convert every SAConv in the model to its deploy form. Returns the number of converted layers."""
    n = 0
    for m in model.modules():
        if isinstance(m, SAConv) and not m.static:
            m.to_static()
            n += 1
    return n


class SAConvTrainer(DetectionTrainer):
    """Load the pretrained weights into the base model first, then insert SAConv initialised from those weights."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_saconv(model)
        if RANK in {-1, 0}:
            print(f"Paper 3 backbone: SAConv at layers {SACONV_LAYERS}")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model

In [ ]:
import importlib, saconv_backbone
importlib.reload(saconv_backbone)
from saconv_backbone import add_saconv, to_static, SAConvTrainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

def fused_params_gflops(model, imgsz=640):
    """Params and GFLOPs of the deploy model (SAConv in static form, Conv+BN fused)."""
    m = deepcopy(model).float().cpu().eval()
    to_static(m)
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# structure check (untrained): base vs Paper 3 backbone
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False)
sac = add_saconv(DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False))
sac.eval()
x = torch.rand(1, 3, 640, 640)
with torch.no_grad():
    y_train_form = sac(x)[0]
    stat = deepcopy(sac); to_static(stat)
    y_static = stat(x)[0]
print("layers 3, 5, 7:", [type(sac.model[i]).__name__ for i in (3, 5, 7)])
print("static form gives the same output:", torch.allclose(y_train_form, y_static, atol=1e-4))
for tag, m in (("base", base), ("Paper 3 backbone", sac)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:17s} Params {pm:.3f} M | GFLOPs {gf:.2f}")

## 4. Training

**Step A:** keep `EPOCHS = 1`. **Step B:** change it to `EPOCHS = 100` and run the notebook again (only if step A passes, section 7).
All other settings are the same as in the earlier notebooks.

In [ ]:
from ultralytics import YOLO

EPOCHS  = 1      # Step A = 1 (export + Pi speed check) | Step B = 100 (real training)
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name)
VARIANTS = [
    ("SAConv Backbone (Paper 3)", "bb_saconv"),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

print(f"Step {'A (1 epoch: export + speed check)' if EPOCHS == 1 else 'B (full training)'}")
for label, run in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO("yolo26n.yaml")
    model.train(trainer=SAConvTrainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()

## 5. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the deploy model (SAConv in static form, Conv+BN fused).
In step A (1 epoch) Precision / Recall / mAP are very low — that is expected; do not use them in the paper.

In [ ]:
import saconv_backbone  # must be imported before loading an SAConv checkpoint

rows = []
for label, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
print(f"(trained for {EPOCHS} epoch{'s' if EPOCHS > 1 else ''})")
METRICS.drop(columns="run")

## 6. Package for Raspberry Pi 5 and Jetson

Creates `paper3_backbone_edge_bench.zip` in the **Output** tab with:

| Item | Used on |
|---|---|
| `bb_saconv_ncnn_model/` (NCNN model, imgsz 640, SAConv in static form) | Raspberry Pi 5 — **only if the export succeeds** |
| `bb_saconv.pt` | Jetson |
| `bench_images/` (50 real test images, seed 0) | both |
| `saconv_backbone.py` | Jetson (needed to load the SAConv `.pt`) |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

The cell prints clearly whether the NCNN export succeeded (✅) or failed (❌).
Both scripts: batch 1, imgsz 640, cycle through the 50 real test images (loaded into memory first),
100 warm-up runs, then 1000 timed runs end-to-end (preprocessing + inference + postprocessing), and print the **median** ms and FPS.
Unzip into its own folder, e.g. `python -m zipfile -e paper3_backbone_edge_bench.zip paper3_backbone_edge_bench`.

In [ ]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper3_backbone_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("saconv_backbone.py", EDGE / "saconv_backbone.py")

# 50 real test images (fixed seed, so every model and device uses the same images)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

from saconv_backbone import to_static

NCNN_OK = False
for _, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")  # Jetson: the trained model as is
    print(f"{run:12s} -> {run}.pt")
    try:
        yolo = YOLO(str(w))
        n = to_static(yolo.model)  # shared-weight branches -> two plain convs, so NCNN sees standard layers
        p = yolo.export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
        shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
        NCNN_OK = True
        print(f"\n✅ NCNN export OK ({n} SAConv layers converted) -> {run}_ncnn_model/  -> run pi5_bench.py on the Pi")
    except Exception as e:
        NCNN_OK = False
        print(f"\n❌ NCNN export FAILED: {type(e).__name__}: {e}")
        print("   SAConv cannot run on the Raspberry Pi 5 with NCNN -> drop it (Jetson package is still created).")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import saconv_backbone  # needed to load the SAConv checkpoint
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper3_backbone_edge_bench"), "zip", EDGE)
print("\nDownload paper3_backbone_edge_bench.zip from the Output tab.")

### How to run on the devices

**Raspberry Pi 5** (same Pi and same conditions as the baseline measurement)
1. Activate the Python environment that has `ultralytics==8.4.137` and `ncnn`.
2. `echo performance | sudo tee /sys/devices/system/cpu/cpu*/scaling_governor`
3. Go into the unzipped folder and run `python pi5_bench.py`.
4. Run it 3 times with a 2–3 minute pause; use the middle value. Check `vcgencmd measure_temp` and `vcgencmd get_throttled` (`throttled=0x0` = OK).

**Jetson**
1. NVIDIA Jetson PyTorch + torchvision for your JetPack, then `ultralytics==8.4.137`.
2. `sudo nvpmodel -m 0` (check with `sudo nvpmodel -q`) and `sudo jetson_clocks`.
3. Go into the unzipped folder and run `python jetson_bench.py`. Run 3 times; use the middle value.

## 7. Step A decision (after `EPOCHS = 1`)

| Result | Decision |
|---|---|
| ❌ NCNN export failed (section 6) | **Drop SAConv**: it cannot run on the Raspberry Pi 5 with NCNN |
| ✅ Export OK, but Pi FPS far below the baseline (e.g. under 10; baseline = 13.38) | **Drop SAConv**: the speed loss is too large |
| ✅ Export OK and Pi FPS close to or above the baseline | **Go to step B**: set `EPOCHS = 100` in section 4 and run the whole notebook again, then measure the FPS again on the new zip |

## 8. Final backbone table

Use this after **step B**. The baseline row is already filled in. Type the median FPS of the SAConv model
(and the baseline Jetson FPS when you have it), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 5 first in the same session (it fills `METRICS`).

In [ ]:
# baseline values (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_saconv": None}
FPS_JETSON = {"bb_saconv": None}

if EPOCHS == 1:
    print("⚠️ This is the 1-epoch step A model: accuracy values are not final.")
table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

## 9. Zip all results

Run this last. It creates **`paper3_backbone_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper3_backbone_edge_bench/` | NCNN (if exported) + `.pt` model, 50 bench images, `saconv_backbone.py`, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `best.pt`, renamed `bb_saconv_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |

In [ ]:
import zipfile

ALL_ZIP = WORK / "paper3_backbone_all_results.zip"
folders = [WORK / "paper3_backbone_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")